In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
%%capture
!pip install ultralytics --quiet
!pip install transformers accelerate datasets --quiet
!pip install albumentations --quiet
!pip install segmentation-models-pytorch --quiet
!pip install faiss-cpu --quiet
!pip install einops --quiet

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

import albumentations as A
from albumentations.pytorch import ToTensorV2

from ultralytics import YOLO

print("All imports successful.")

In [ ]:
SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(SEED)
print(f"All random seeds fixed to {SEED}")

In [ ]:
# Cell 5
# ── Dataset Paths ──────────────────────────────────────────────────
BASE_DIR        = Path("/kaggle/input/datasets/solesensei/solesensei_bdd100k")

# Detection (100k images + labels)
DET_IMG_TRAIN   = BASE_DIR / "bdd100k" / "bdd100k" / "images" / "100k" / "train"
DET_IMG_VAL     = BASE_DIR / "bdd100k" / "bdd100k" / "images" / "100k" / "val"
DET_LABEL_DIR   = BASE_DIR / "bdd100k_labels_release" / "bdd100k" / "labels"

# Segmentation (10k images + masks)
SEG_IMG_TRAIN   = BASE_DIR / "bdd100k_seg" / "bdd100k" / "seg" / "images" / "train"
SEG_IMG_VAL     = BASE_DIR / "bdd100k_seg" / "bdd100k" / "seg" / "images" / "val"
SEG_MASK_TRAIN  = BASE_DIR / "bdd100k_seg" / "bdd100k" / "seg" / "labels" / "train"
SEG_MASK_VAL    = BASE_DIR / "bdd100k_seg" / "bdd100k" / "seg" / "labels" / "val"
SEG_COLOR_TRAIN = BASE_DIR / "bdd100k_seg" / "bdd100k" / "seg" / "color_labels" / "train"
SEG_COLOR_VAL   = BASE_DIR / "bdd100k_seg" / "bdd100k" / "seg" / "color_labels" / "val"

# Output directories
OUT_DIR         = Path("/kaggle/working")
YOLO_DIR        = OUT_DIR / "yolo_dataset"
CKPT_DIR        = OUT_DIR / "checkpoints"
RESULTS_DIR     = OUT_DIR / "results"

for d in [YOLO_DIR, CKPT_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── 10 Selected Classes ────────────────────────────────────────────
# Mandatory: car, pedestrian, traffic light, lane marking
# Road hazard proxy: drivable area (BDD100K has no pothole/crack annotations)
# Remaining: chosen by frequency + road safety relevance
SELECTED_CLASSES = [
    "car",            # 0 - mandatory
    "person",         # 1 - mandatory (NOT "pedestrian" — this is the actual name)
    "traffic light",  # 2 - mandatory
    "bike",           # 3 - road hazard proxy / vulnerable user (lane marking is poly2d, not box2d)
    "truck",          # 4 - high frequency, safety critical
    "bus",            # 5 - high frequency, safety critical
    "rider",          # 6 - vulnerable road user
    "motor",          # 7 - vulnerable road user
    "traffic sign",   # 8 - road safety
    "train",          # 9 - vehicle category restriction class
]

CLASS2ID = {cls: idx for idx, cls in enumerate(SELECTED_CLASSES)}
ID2CLASS  = {idx: cls for cls, idx in CLASS2ID.items()}
NUM_CLASSES = len(SELECTED_CLASSES)

# ── YOLO Hyperparameters ───────────────────────────────────────────
YOLO_MODEL    = "yolo11n.pt"
IMG_SIZE      = 640
BATCH_SIZE    = 16
EPOCHS        = 50
LR0           = 0.01
MOMENTUM      = 0.937
PATIENCE      = 10

# ── Segmentation Hyperparameters ──────────────────────────────────
SEG_IMG_SIZE    = 512
SEG_BATCH_SIZE  = 8
SEG_EPOCHS      = 50
SEG_LR          = 6e-5
UNET_LR         = 1e-3

# ImageNet normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=== Global Config ===")
print(f"Device        : {DEVICE}")
print(f"Num classes   : {NUM_CLASSES}")
print(f"Classes       : {SELECTED_CLASSES}")
print(f"YOLO model    : {YOLO_MODEL}")
print(f"Batch size    : {BATCH_SIZE}")
print(f"Epochs        : {EPOCHS}")
print(f"Output dir    : {OUT_DIR}")

In [ ]:
#Cell 6
paths_to_check = {
    "Det images (train)"  : DET_IMG_TRAIN,
    "Det images (val)"    : DET_IMG_VAL,
    "Det labels dir"      : DET_LABEL_DIR,
    "Seg images (train)"  : SEG_IMG_TRAIN,
    "Seg images (val)"    : SEG_IMG_VAL,
    "Seg masks (train)"   : SEG_MASK_TRAIN,
    "Seg masks (val)"     : SEG_MASK_VAL,
    "Seg color (train)"   : SEG_COLOR_TRAIN,
    "Seg color (val)"     : SEG_COLOR_VAL,
}

all_ok = True
for name, path in paths_to_check.items():
    exists = path.exists()
    status = "✅" if exists else "❌ NOT FOUND"
    print(f"{status}  {name}: {path}")
    if not exists:
        all_ok = False

print()
print("All paths OK ✅" if all_ok else "⚠️  Fix the paths above before proceeding.")

# Also peek inside label dir to find the JSON files
print("\n── Contents of DET_LABEL_DIR ──")
if DET_LABEL_DIR.exists():
    for item in sorted(DET_LABEL_DIR.iterdir()):
        print(f"  {item.name}")
else:
    print("  Directory not found.")

**Phase 1: EDA and Visualization**

In [ ]:
#1.1.1
# ── Find detection label JSONs ─────────────────────────────────────
print("── Contents of DET_LABEL_DIR ──")
for item in sorted(DET_LABEL_DIR.iterdir()):
    print(f"  {item.name}")

# Load train labels
det_label_train_path = DET_LABEL_DIR / "bdd100k_labels_images_train.json"
det_label_val_path   = DET_LABEL_DIR / "bdd100k_labels_images_val.json"

print("\nLoading detection labels...")
with open(det_label_train_path, "r") as f:
    det_train_raw = json.load(f)

with open(det_label_val_path, "r") as f:
    det_val_raw = json.load(f)

print(f"Train samples : {len(det_train_raw):,}")
print(f"Val samples   : {len(det_val_raw):,}")
print(f"\nSample entry keys: {list(det_train_raw[0].keys())}")
print(f"\nSample entry:\n{json.dumps(det_train_raw[0], indent=2)}")

In [ ]:
#1.1.2
def parse_detection_json(raw_data, split_name="train"):
    """
    Parse BDD100K detection JSON into a flat DataFrame.
    Each row = one bounding box annotation.
    """
    records = []
    for entry in tqdm(raw_data, desc=f"Parsing {split_name}"):
        img_name  = entry.get("name", "")
        attrs     = entry.get("attributes", {})
        weather   = attrs.get("weather", "unknown")
        timeofday = attrs.get("timeofday", "unknown")
        scene     = attrs.get("scene", "unknown")
        labels    = entry.get("labels", [])

        if labels is None:
            labels = []

        for label in labels:
            category = label.get("category", "unknown")
            box2d    = label.get("box2d", None)

            if box2d is None:
                continue  # skip non-box labels (e.g. lane polylines)

            x1 = box2d["x1"]
            y1 = box2d["y1"]
            x2 = box2d["x2"]
            y2 = box2d["y2"]
            w  = x2 - x1
            h  = y2 - y1
            area = w * h

            records.append({
                "image"     : img_name,
                "split"     : split_name,
                "category"  : category,
                "weather"   : weather,
                "timeofday" : timeofday,
                "scene"     : scene,
                "x1"        : x1,
                "y1"        : y1,
                "x2"        : x2,
                "y2"        : y2,
                "width"     : w,
                "height"    : h,
                "area"      : area,
            })

    return pd.DataFrame(records)


df_train = parse_detection_json(det_train_raw, "train")
df_val   = parse_detection_json(det_val_raw,   "val")
df_all   = pd.concat([df_train, df_val], ignore_index=True)

print(f"\nTotal annotations : {len(df_all):,}")
print(f"Train annotations : {len(df_train):,}")
print(f"Val annotations   : {len(df_val):,}")
print(f"\nUnique categories ({df_all['category'].nunique()}):")
print(sorted(df_all["category"].unique()))
print(f"\nWeather values    : {sorted(df_all['weather'].unique())}")
print(f"Time of day values: {sorted(df_all['timeofday'].unique())}")
print(f"\nDataFrame shape   : {df_all.shape}")
df_train.head(3)

In [ ]:
#1.1.3
def parse_seg_metadata(img_dir, mask_dir, split_name="train"):
    """
    Build a DataFrame of segmentation image paths + mask paths.
    Mask files are named: <stem>_train_id.png
    Image files are named: <stem>.jpg
    """
    img_dir  = Path(img_dir)
    mask_dir = Path(mask_dir)

    img_files  = sorted(img_dir.glob("*.jpg"))
    # Build lookup: stem -> full mask path (suffix is _train_id.png)
    mask_files = {f.stem.replace("_train_id", ""): f for f in mask_dir.glob("*_train_id.png")}

    records = []
    for img_path in img_files:
        stem      = img_path.stem
        mask_path = mask_files.get(stem, None)
        records.append({
            "image_name" : stem,
            "split"      : split_name,
            "img_path"   : str(img_path),
            "mask_path"  : str(mask_path) if mask_path else None,
            "has_mask"   : mask_path is not None,
        })

    df = pd.DataFrame(records)
    return df


df_seg_train = parse_seg_metadata(SEG_IMG_TRAIN, SEG_MASK_TRAIN, "train")
df_seg_val   = parse_seg_metadata(SEG_IMG_VAL,   SEG_MASK_VAL,   "val")
df_seg_all   = pd.concat([df_seg_train, df_seg_val], ignore_index=True)

print(f"Seg train images    : {len(df_seg_train):,}")
print(f"Seg val images      : {len(df_seg_val):,}")
print(f"Total seg images    : {len(df_seg_all):,}")
print(f"Images with masks   : {df_seg_all['has_mask'].sum():,}")
print(f"Images missing masks: {(~df_seg_all['has_mask']).sum():,}")

# Verify a mask actually loads correctly
sample = df_seg_train[df_seg_train["has_mask"]].iloc[0]
mask   = np.array(Image.open(sample["mask_path"]))
print(f"\nSample mask shape  : {mask.shape}")
print(f"Unique class IDs   : {np.unique(mask)}")
print(f"Sample image path  : {sample['img_path']}")
print(f"Sample mask path   : {sample['mask_path']}")
df_seg_train.head(3)

In [ ]:
#1.1.4
# ── Filter to our 10 selected classes ─────────────────────────────
df_train_10 = df_train[df_train["category"].isin(SELECTED_CLASSES)].copy()
df_val_10   = df_val[df_val["category"].isin(SELECTED_CLASSES)].copy()
df_all_10   = pd.concat([df_train_10, df_val_10], ignore_index=True)

# Add integer class ID
df_train_10["class_id"] = df_train_10["category"].map(CLASS2ID)
df_val_10["class_id"]   = df_val_10["category"].map(CLASS2ID)
df_all_10["class_id"]   = df_all_10["category"].map(CLASS2ID)

print("=== Class Filtering Summary ===")
print(f"Total annotations (all classes) : {len(df_all):,}")
print(f"Total annotations (10 classes)  : {len(df_all_10):,}")
print(f"Retained                        : {len(df_all_10)/len(df_all)*100:.1f}%")
print(f"\nPer-class annotation counts (train):")
print(df_train_10["category"].value_counts().to_string())

# Document filtering decisions
print("\n=== Filtering Decision Log ===")
all_cats = sorted(df_train["category"].unique())
for cat in all_cats:
    count  = (df_train["category"] == cat).sum()
    kept   = "✅ KEPT" if cat in SELECTED_CLASSES else "❌ DROPPED"
    reason = ""
    if cat not in SELECTED_CLASSES:
        reason = "— low frequency / redundant"
    print(f"  {kept}  {cat:<20} ({count:>7,} annotations) {reason}")

In [ ]:
#1.1.5
def show_sample_images(img_dir, n=4, title="Sample Images"):
    img_dir  = Path(img_dir)
    samples  = sorted(img_dir.glob("*.jpg"))[:n]

    fig, axes = plt.subplots(1, n, figsize=(20, 4))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for ax, img_path in zip(axes, samples):
        img = Image.open(img_path)
        ax.imshow(img)
        ax.set_title(img_path.name, fontsize=7)
        ax.axis("off")

    plt.tight_layout()
    plt.show()
    print(f"Image size: {img.size} (W x H)")


show_sample_images(DET_IMG_TRAIN,  n=4, title="Detection — 100k Train Samples")
show_sample_images(SEG_IMG_TRAIN,  n=4, title="Segmentation — 10k Train Samples")

In [ ]:
#1.1.6
# BDD100K standard segmentation class mapping (train_id -> class name)
BDD_SEG_CLASSES = {
    0:   "road",
    1:   "sidewalk",
    2:   "building",
    3:   "wall",
    4:   "fence",
    5:   "pole",
    6:   "traffic light",
    7:   "traffic sign",
    8:   "vegetation",
    9:   "terrain",
    10:  "sky",
    11:  "person",
    12:  "rider",
    13:  "car",
    14:  "truck",
    15:  "bus",
    16:  "train",
    17:  "motor",
    18:  "bike",
    255: "unlabeled / ignore",
}

# Check which classes appear in our sample mask
sample_ids = [0, 2, 4, 5, 6, 7, 8, 10, 11, 13, 255]
print("Classes present in sample mask:")
for cid in sample_ids:
    print(f"  ID {cid:>3} → {BDD_SEG_CLASSES.get(cid, 'unknown')}")

# Define our 10 segmentation classes (mapped from BDD train_ids)
# We pick classes that are: (a) present in masks, (b) road-safety relevant
SEG_CLASS_MAP = {
    0:  0,   # road
    11: 1,   # person
    12: 1,   # rider → merged into person (both are vulnerable road users)
    6:  2,   # traffic light
    7:  3,   # traffic sign
    13: 4,   # car
    14: 5,   # truck
    15: 6,   # bus
    16: 7,   # train
    17: 8,   # motor
    18: 9,   # bike
}

SEG_CLASSES = [
    "road", "person", "traffic light", "traffic sign",
    "car", "truck", "bus", "train", "motor", "bike"
]

print(f"\nSegmentation classes ({len(SEG_CLASSES)}):")
for i, name in enumerate(SEG_CLASSES):
    print(f"  {i} → {name}")

print("\nNote: 'lane marking' appears as part of 'road' class in seg masks.")
print("      255 = unlabeled/ignore — excluded from loss computation.")

In [ ]:
#1.2.1
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle("Category Distribution — BDD100K Detection Annotations", fontsize=14, fontweight="bold")

# All classes
cat_counts_all = df_train["category"].value_counts()
axes[0].bar(cat_counts_all.index, cat_counts_all.values, color="steelblue", edgecolor="black")
axes[0].set_title("All Classes (Train Split)")
axes[0].set_xlabel("Category")
axes[0].set_ylabel("Annotation Count")
axes[0].tick_params(axis="x", rotation=45)
for i, v in enumerate(cat_counts_all.values):
    axes[0].text(i, v + 1000, f"{v:,}", ha="center", fontsize=8)

# Selected 10 classes only
cat_counts_10 = df_train_10["category"].value_counts()
colors = plt.cm.tab10(np.linspace(0, 1, len(cat_counts_10)))
axes[1].bar(cat_counts_10.index, cat_counts_10.values, color=colors, edgecolor="black")
axes[1].set_title("Selected 10 Classes (Train Split)")
axes[1].set_xlabel("Category")
axes[1].set_ylabel("Annotation Count")
axes[1].tick_params(axis="x", rotation=45)
for i, v in enumerate(cat_counts_10.values):
    axes[1].text(i, v + 500, f"{v:,}", ha="center", fontsize=8)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_category_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: eda_category_distribution.png")

In [ ]:
#1.2.2
print("Sampling images per weather condition for intensity analysis...")

weather_conditions = ["clear", "rainy", "snowy", "foggy", "overcast", "partly cloudy"]
weather_colors     = ["gold", "royalblue", "lightcyan", "lightgray", "slategray", "plum"]
N_SAMPLES          = 100

weather_intensities = {}

for weather in weather_conditions:
    imgs_for_weather = df_train[df_train["weather"] == weather]["image"].unique()
    sampled          = np.random.choice(imgs_for_weather,
                                        size=min(N_SAMPLES, len(imgs_for_weather)),
                                        replace=False)
    intensities = []
    failed      = 0
    for img_name in sampled:
        img_path = DET_IMG_TRAIN / img_name
        if not img_path.exists():
            failed += 1
            continue
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            failed += 1
            continue
        intensities.extend(img.flatten()[::10].tolist())

    weather_intensities[weather] = intensities
    print(f"  {weather:<15}: {len(imgs_for_weather):>5} images | "
          f"sampled {len(sampled)} | failed {failed} | pixels: {len(intensities):,}")

# Plot per-weather histograms
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Pixel Intensity Distribution by Weather Condition", fontsize=14, fontweight="bold")

for ax, weather, color in zip(axes.flatten(), weather_conditions, weather_colors):
    intensities = weather_intensities[weather]
    if len(intensities) == 0:
        ax.text(0.5, 0.5, f"No valid images\nfor '{weather}'",
                ha="center", va="center", transform=ax.transAxes, fontsize=12)
        ax.set_title(f"{weather.capitalize()} (no data)")
        continue
    ax.hist(intensities, bins=64, color=color, edgecolor="black", alpha=0.8, density=True)
    mean_val = np.mean(intensities)
    ax.axvline(mean_val, color="red", linestyle="--",
               linewidth=1.5, label=f"Mean={mean_val:.1f}")
    ax.set_title(f"{weather.capitalize()} (n={len(intensities):,} px)")
    ax.set_xlabel("Pixel Intensity (0–255)")
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_weather_intensity.png", dpi=150, bbox_inches="tight")
plt.show()

# Overlay plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_title("Overlapping Pixel Intensity Distributions by Weather Condition",
             fontsize=13, fontweight="bold")
for weather, color in zip(weather_conditions, weather_colors):
    intensities = weather_intensities[weather]
    if len(intensities) == 0:
        continue
    ax.hist(intensities, bins=64, color=color, alpha=0.5,
            density=True, label=weather.capitalize(), edgecolor="none")
ax.set_xlabel("Pixel Intensity (0–255)")
ax.set_ylabel("Density")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_weather_intensity_overlay.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: eda_weather_intensity.png + eda_weather_intensity_overlay.png")

In [ ]:
#1.2.3
# Time-of-day image counts
tod_image_counts = df_train.groupby("timeofday")["image"].nunique()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Time-of-Day Analysis", fontsize=14, fontweight="bold")

# Bar chart — image count
axes[0].bar(tod_image_counts.index, tod_image_counts.values,
            color=["orange", "navy", "purple", "gray"], edgecolor="black")
axes[0].set_title("Image Count per Time of Day")
axes[0].set_xlabel("Time of Day")
axes[0].set_ylabel("Number of Images")
for i, v in enumerate(tod_image_counts.values):
    axes[0].text(i, v + 50, f"{v:,}", ha="center", fontsize=9)

# Pie chart
axes[1].pie(tod_image_counts.values, labels=tod_image_counts.index,
            autopct="%1.1f%%", colors=["orange", "navy", "purple", "gray"],
            startangle=90)
axes[1].set_title("Time-of-Day Distribution (%)")

# Brightness per time of day
print("Sampling brightness per time-of-day (1-2 min)...")
tod_conditions = ["daytime", "night", "dawn/dusk", "undefined"]
tod_colors     = ["orange", "navy", "purple", "gray"]
N_TOD          = 80

tod_brightness = {}
for tod in tod_conditions:
    imgs = df_train[df_train["timeofday"] == tod]["image"].unique()
    sampled = np.random.choice(imgs, size=min(N_TOD, len(imgs)), replace=False)
    brightness = []
    for img_name in sampled:
        img_path = DET_IMG_TRAIN / img_name
        if img_path.exists():
            img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
            if img is not None:
                brightness.append(np.mean(img))
    tod_brightness[tod] = brightness

axes[2].boxplot([tod_brightness[t] for t in tod_conditions],
                labels=[t.replace("/", "/\n") for t in tod_conditions],
                patch_artist=True,
                boxprops=dict(facecolor="lightblue"),
                medianprops=dict(color="red", linewidth=2))
axes[2].set_title("Image Brightness by Time of Day")
axes[2].set_xlabel("Time of Day")
axes[2].set_ylabel("Mean Pixel Intensity")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_timeofday_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: eda_timeofday_analysis.png")

# Print brightness stats
print("\nBrightness Statistics:")
for tod in tod_conditions:
    b = tod_brightness[tod]
    if b:
        print(f"  {tod:<12}: mean={np.mean(b):.1f}  std={np.std(b):.1f}  "
              f"min={np.min(b):.1f}  max={np.max(b):.1f}")

In [ ]:
#1.2.4
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle("Class Imbalance Analysis — Selected 10 Classes", fontsize=14, fontweight="bold")

cat_counts = df_train_10["category"].value_counts()
total      = cat_counts.sum()

# Absolute counts with imbalance ratio
colors = ["#2ecc71" if v > total / len(cat_counts) else "#e74c3c"
          for v in cat_counts.values]
bars = axes[0].barh(cat_counts.index, cat_counts.values, color=colors, edgecolor="black")
axes[0].set_title("Annotation Counts (Green=Over-represented, Red=Under-represented)")
axes[0].set_xlabel("Annotation Count")
for bar, v in zip(bars, cat_counts.values):
    axes[0].text(v + 1000, bar.get_y() + bar.get_height() / 2,
                 f"{v:,}", va="center", fontsize=9)
mean_line = total / len(cat_counts)
axes[0].axvline(mean_line, color="black", linestyle="--",
                linewidth=1.5, label=f"Mean = {mean_line:,.0f}")
axes[0].legend()

# Log scale for better visibility
axes[1].barh(cat_counts.index, cat_counts.values, color=colors, edgecolor="black")
axes[1].set_xscale("log")
axes[1].set_title("Annotation Counts (Log Scale)")
axes[1].set_xlabel("Annotation Count (log)")
for bar, v in zip(bars, cat_counts.values):
    axes[1].text(v * 1.05, bar.get_y() + bar.get_height() / 2,
                 f"{v:,}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_class_imbalance.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nClass Imbalance Summary:")
print(f"{'Class':<15} {'Count':>10} {'% of Total':>12} {'Imbalance Ratio':>16}")
print("-" * 55)
for cls, cnt in cat_counts.items():
    pct   = cnt / total * 100
    ratio = cnt / mean_line
    flag  = "▲ OVER" if ratio > 1 else "▼ UNDER"
    print(f"{cls:<15} {cnt:>10,} {pct:>11.1f}% {ratio:>12.2f}x  {flag}")
print("Saved: eda_class_imbalance.png")

In [ ]:
#1.2.5
def draw_bboxes_on_image(img_path, annotations_df, class2color):
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    for _, row in annotations_df.iterrows():
        x1, y1, x2, y2 = int(row.x1), int(row.y1), int(row.x2), int(row.y2)
        cat   = row.category
        color = class2color.get(cat, (255, 255, 255))
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        label = cat
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(img, (x1, y1 - th - 4), (x1 + tw, y1), color, -1)
        cv2.putText(img, label, (x1, y1 - 2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
    return img

# Color per class
np.random.seed(SEED)
CLASS_COLORS = {
    cls: tuple(int(c) for c in np.random.randint(50, 230, 3))
    for cls in SELECTED_CLASSES
}

# Pick images that have >= 3 boxes AND the file actually exists
rich_images = (df_train_10.groupby("image")
               .filter(lambda x: len(x) >= 3)["image"].unique())

# Filter to only images that exist on disk
valid_images = [img for img in rich_images
                if (DET_IMG_TRAIN / img).exists()]
print(f"Valid images available: {len(valid_images):,}")

sample_imgs = np.random.choice(valid_images, size=10, replace=False)

fig, axes = plt.subplots(2, 5, figsize=(25, 10))
fig.suptitle("10 Annotated Example Images — BDD100K Detection (Selected 10 Classes)",
             fontsize=14, fontweight="bold")

for ax, img_name in zip(axes.flatten(), sample_imgs):
    img_path   = DET_IMG_TRAIN / img_name
    ann_subset = df_train_10[df_train_10["image"] == img_name]
    annotated  = draw_bboxes_on_image(img_path, ann_subset, CLASS_COLORS)
    if annotated is None:
        ax.text(0.5, 0.5, "Read error", ha="center", va="center",
                transform=ax.transAxes)
        ax.axis("off")
        continue
    ax.imshow(annotated)
    ax.set_title(f"{img_name[:20]}\n({len(ann_subset)} boxes)", fontsize=8)
    ax.axis("off")

handles = [patches.Patch(color=np.array(v) / 255, label=k)
           for k, v in CLASS_COLORS.items()]
fig.legend(handles=handles, loc="lower center", ncol=5,
           fontsize=9, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_annotated_examples.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: eda_annotated_examples.png")

In [ ]:
#1.2.6
def overlay_seg_mask(img_path, color_mask_path):
    """Overlay color segmentation mask on image."""
    img        = np.array(Image.open(img_path).convert("RGB"))
    color_mask = np.array(Image.open(color_mask_path).convert("RGB"))
    overlay    = cv2.addWeighted(img, 0.6, color_mask, 0.4, 0)
    return img, color_mask, overlay


# Sample 5 images from segmentation set for overlay visualization
seg_samples = df_seg_train[df_seg_train["has_mask"]].sample(5, random_state=SEED)

# Get corresponding color label paths
color_mask_files = {
    f.stem.replace("_train_color", ""): f
    for f in SEG_COLOR_TRAIN.glob("*_train_color.png")
}

fig, axes = plt.subplots(5, 3, figsize=(18, 22))
fig.suptitle("Segmentation Examples — Image | Color Mask | Overlay",
             fontsize=14, fontweight="bold")

col_titles = ["Original Image", "Segmentation Mask", "Overlay"]
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=12, fontweight="bold")

for i, (_, row) in enumerate(seg_samples.iterrows()):
    img_path        = Path(row["img_path"])
    color_mask_path = color_mask_files.get(row["image_name"], None)

    if color_mask_path is None:
        print(f"  No color mask for {row['image_name']}, skipping.")
        continue

    img, color_mask, overlay = overlay_seg_mask(img_path, color_mask_path)

    axes[i][0].imshow(img)
    axes[i][0].axis("off")
    axes[i][1].imshow(color_mask)
    axes[i][1].axis("off")
    axes[i][2].imshow(overlay)
    axes[i][2].axis("off")
    axes[i][0].set_ylabel(row["image_name"][:15], fontsize=8, rotation=0,
                          labelpad=60, va="center")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "eda_segmentation_overlays.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: eda_segmentation_overlays.png")

In [ ]:
#1.2.7
justification_data = {
    "Class"        : SELECTED_CLASSES,
    "Train Count"  : [df_train_10[df_train_10["category"] == c].shape[0]
                      for c in SELECTED_CLASSES],
    "% of Total"   : [round(df_train_10[df_train_10["category"] == c].shape[0]
                      / len(df_train_10) * 100, 2) for c in SELECTED_CLASSES],
    "Mandatory"    : ["✅" if c in ["car", "person", "traffic light"] else
                      "⚠️ proxy" if c == "bike" else "—"
                      for c in SELECTED_CLASSES],
    "Safety Role"  : [
        "Primary road agent",
        "Vulnerable road user (mandatory)",
        "Intersection control (mandatory)",
        "Road hazard proxy — vulnerable 2-wheeler",
        "Heavy vehicle — collision risk",
        "Heavy vehicle — collision risk",
        "Mixed traffic — high risk",
        "Motorized 2-wheeler — high risk",
        "Speed/regulation enforcement",
        "Rail vehicle — restricted zones",
    ],
}

df_justification = pd.DataFrame(justification_data)
print("=== 10-Class Subset Justification ===\n")
print(df_justification.to_string(index=False))

print("""
=== Filtering Decisions ===
DROPPED 'lane'         : Annotated as poly2d polylines — no bounding boxes available.
                         Lane markings ARE covered in the segmentation task (road class).
DROPPED 'drivable area': Annotated as poly2d polygons — incompatible with YOLO bbox format.
DROPPED 'train'        : Only 136 annotations — statistically insufficient for training.
NOTE    'pedestrian'   : BDD100K uses 'person' as the actual category name.
NOTE    'bicycle'      : BDD100K uses 'bike' as the actual category name.
NOTE    'pothole/crack': Not annotated in BDD100K. Road surface hazards are represented
                         via the segmentation pipeline's road class boundary analysis.
""")

df_justification.to_csv(RESULTS_DIR / "eda_class_justification.csv", index=False)
print("Saved: eda_class_justification.csv")

In [ ]:
#1.2.8
eda_summary = {
    "total_train_images"          : df_train["image"].nunique(),
    "total_val_images"            : df_val["image"].nunique(),
    "total_train_annotations"     : len(df_train),
    "total_10class_annotations"   : len(df_train_10),
    "seg_train_images"            : len(df_seg_train),
    "seg_val_images"              : len(df_seg_val),
    "num_weather_conditions"      : df_train["weather"].nunique(),
    "num_timeofday_conditions"    : df_train["timeofday"].nunique(),
    "most_common_class"           : df_train_10["category"].value_counts().idxmax(),
    "least_common_class"          : df_train_10["category"].value_counts().idxmin(),
    "imbalance_ratio"             : round(
                                        df_train_10["category"].value_counts().max() /
                                        df_train_10["category"].value_counts().min(), 1),
}

df_eda_summary = pd.DataFrame([eda_summary]).T.rename(columns={0: "value"})
print("=== EDA Summary ===")
print(df_eda_summary.to_string())
df_eda_summary.to_csv(RESULTS_DIR / "eda_summary.csv")
print("\nSaved: eda_summary.csv")

**Phase 2 — YOLOv11 Object Detection**

In [ ]:
#2.1.1
# BDD100K images are split across trainA/trainB subfolders
# We need to find ALL images across all subfolders

def discover_all_images(root_dir):
    """Recursively find all jpg images under root_dir."""
    root_dir = Path(root_dir)
    all_imgs = list(root_dir.rglob("*.jpg"))
    print(f"Found {len(all_imgs):,} images under {root_dir}")
    return all_imgs

train_imgs_all = discover_all_images(DET_IMG_TRAIN)
val_imgs_all   = discover_all_images(DET_IMG_VAL)

# Build lookup: image stem -> full path
train_img_lookup = {p.name: p for p in train_imgs_all}
val_img_lookup   = {p.name: p for p in val_imgs_all}

print(f"\nTrain image lookup size : {len(train_img_lookup):,}")
print(f"Val image lookup size   : {len(val_img_lookup):,}")

# Verify a known image resolves correctly
sample_name = df_train_10.iloc[0]["image"]
resolved    = train_img_lookup.get(sample_name, None)
print(f"\nSample lookup test:")
print(f"  Image name : {sample_name}")
print(f"  Resolved   : {resolved}")

In [ ]:
#2.1.2
# YOLO expects this structure:
# yolo_dataset/
#   images/train/   images/val/   images/test/
#   labels/train/   labels/val/   labels/test/

for split in ["train", "val", "test"]:
    (YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

print("YOLO directory structure created:")
for split in ["train", "val", "test"]:
    print(f"  {YOLO_DIR}/images/{split}/")
    print(f"  {YOLO_DIR}/labels/{split}/")

In [ ]:
#2.1.3
import shutil

# ── Subset sizes ───────────────────────────────────────────────────
N_TRAIN = 20000   # out of 70,000
N_VAL   = 2000    # out of 8,000
N_TEST  = 500     # out of 2,000

def bbox_to_yolo(x1, y1, x2, y2, img_w=1280, img_h=720):
    """Convert absolute xyxy bbox to YOLO normalized xywh format."""
    cx = (x1 + x2) / 2.0 / img_w
    cy = (y1 + y2) / 2.0 / img_h
    w  = (x2 - x1) / img_w
    h  = (y2 - y1) / img_h
    # Clamp to [0, 1]
    cx = max(0.0, min(1.0, cx))
    cy = max(0.0, min(1.0, cy))
    w  = max(0.0, min(1.0, w))
    h  = max(0.0, min(1.0, h))
    return cx, cy, w, h

def convert_to_yolo_fast(df, img_lookup, split_name,
                         yolo_img_dir, yolo_lbl_dir,
                         max_images=None, img_w=1280, img_h=720):
    yolo_img_dir = Path(yolo_img_dir)
    yolo_lbl_dir = Path(yolo_lbl_dir)

    images_in_split = df["image"].unique()

    # Shuffle and subset
    np.random.seed(SEED)
    np.random.shuffle(images_in_split)
    if max_images:
        images_in_split = images_in_split[:max_images]

    processed = 0
    skipped   = 0

    for img_name in tqdm(images_in_split, desc=f"Converting {split_name}"):
        src_path = img_lookup.get(img_name, None)
        if src_path is None or not src_path.exists():
            skipped += 1
            continue

        # Write label file
        img_annotations = df[df["image"] == img_name]
        label_lines     = []

        for _, row in img_annotations.iterrows():
            class_id     = row["class_id"]
            cx, cy, w, h = bbox_to_yolo(
                row["x1"], row["y1"], row["x2"], row["y2"],
                img_w=img_w, img_h=img_h
            )
            if w < 1e-4 or h < 1e-4:
                continue
            label_lines.append(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

        if len(label_lines) == 0:
            skipped += 1
            continue

        lbl_path = yolo_lbl_dir / (Path(img_name).stem + ".txt")
        with open(lbl_path, "w") as f:
            f.write("\n".join(label_lines))

        # Symlink image (instant — no copying)
        dst_img = yolo_img_dir / img_name
        if not dst_img.exists():
            os.symlink(src_path, dst_img)

        processed += 1

    return processed, skipped


# Convert splits
train_processed, train_skipped = convert_to_yolo_fast(
    df           = df_train_10,
    img_lookup   = train_img_lookup,
    split_name   = "train",
    yolo_img_dir = YOLO_DIR / "images" / "train",
    yolo_lbl_dir = YOLO_DIR / "labels" / "train",
    max_images   = N_TRAIN,
)

# Val/test from val split
val_imgs_unique  = df_val_10["image"].unique()
np.random.seed(SEED)
np.random.shuffle(val_imgs_unique)

val_imgs_subset  = val_imgs_unique[:N_VAL]
test_imgs_subset = val_imgs_unique[N_VAL: N_VAL + N_TEST]

df_val_yolo  = df_val_10[df_val_10["image"].isin(val_imgs_subset)]
df_test_yolo = df_val_10[df_val_10["image"].isin(test_imgs_subset)]

val_processed, val_skipped = convert_to_yolo_fast(
    df           = df_val_yolo,
    img_lookup   = val_img_lookup,
    split_name   = "val",
    yolo_img_dir = YOLO_DIR / "images" / "val",
    yolo_lbl_dir = YOLO_DIR / "labels" / "val",
    max_images   = N_VAL,
)

test_processed, test_skipped = convert_to_yolo_fast(
    df           = df_test_yolo,
    img_lookup   = val_img_lookup,
    split_name   = "test",
    yolo_img_dir = YOLO_DIR / "images" / "test",
    yolo_lbl_dir = YOLO_DIR / "labels" / "test",
    max_images   = N_TEST,
)

print("\n=== Conversion Summary ===")
print(f"Train — processed: {train_processed:,}  skipped: {train_skipped:,}")
print(f"Val   — processed: {val_processed:,}  skipped: {val_skipped:,}")
print(f"Test  — processed: {test_processed:,}  skipped: {test_skipped:,}")

In [ ]:
#2.1.4
import yaml

data_yaml = {
    "path"  : str(YOLO_DIR),
    "train" : "images/train",
    "val"   : "images/val",
    "test"  : "images/test",
    "nc"    : NUM_CLASSES,
    "names" : SELECTED_CLASSES,
}

yaml_path = YOLO_DIR / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

print("=== data.yaml ===")
with open(yaml_path) as f:
    print(f.read())
print(f"Saved: {yaml_path}")

In [ ]:
#2.1.5
def verify_yolo_split(img_dir, lbl_dir, split_name, n_check=5):
    img_dir = Path(img_dir)
    lbl_dir = Path(lbl_dir)

    img_files = list(img_dir.glob("*.jpg"))
    lbl_files = list(lbl_dir.glob("*.txt"))

    # Check label/image correspondence
    img_stems = {f.stem for f in img_files}
    lbl_stems = {f.stem for f in lbl_files}
    matched   = img_stems & lbl_stems
    img_only  = img_stems - lbl_stems
    lbl_only  = lbl_stems - img_stems

    print(f"\n── {split_name} split ──")
    print(f"  Images       : {len(img_files):,}")
    print(f"  Labels       : {len(lbl_files):,}")
    print(f"  Matched pairs: {len(matched):,}")
    print(f"  Images w/o label: {len(img_only):,}")
    print(f"  Labels w/o image: {len(lbl_only):,}")

    # Spot-check a few label files
    print(f"  Sample label contents:")
    for lbl_path in list(lbl_dir.glob("*.txt"))[:n_check]:
        with open(lbl_path) as f:
            lines = f.readlines()
        print(f"    {lbl_path.name}: {len(lines)} boxes | "
              f"first line: {lines[0].strip() if lines else 'EMPTY'}")

verify_yolo_split(YOLO_DIR / "images" / "train",
                  YOLO_DIR / "labels" / "train", "train")
verify_yolo_split(YOLO_DIR / "images" / "val",
                  YOLO_DIR / "labels" / "val",   "val")
verify_yolo_split(YOLO_DIR / "images" / "test",
                  YOLO_DIR / "labels" / "test",  "test")

In [ ]:
#2.1.6
def visualize_yolo_sample(img_dir, lbl_dir, class_names, class_colors, n=4):
    img_dir = Path(img_dir)
    lbl_dir = Path(lbl_dir)

    img_files = list(img_dir.glob("*.jpg"))[:n]
    fig, axes = plt.subplots(1, n, figsize=(20, 5))
    fig.suptitle("YOLO Format Sanity Check — Bounding Boxes", fontsize=13, fontweight="bold")

    for ax, img_path in zip(axes, img_files):
        img      = cv2.imread(str(img_path))
        if img is None:
            ax.axis("off")
            continue
        img      = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w     = img.shape[:2]
        lbl_path = lbl_dir / (img_path.stem + ".txt")

        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cid, cx, cy, bw, bh = int(parts[0]), *map(float, parts[1:])
                    x1 = int((cx - bw / 2) * w)
                    y1 = int((cy - bh / 2) * h)
                    x2 = int((cx + bw / 2) * w)
                    y2 = int((cy + bh / 2) * h)
                    color = class_colors.get(class_names[cid], (255, 255, 255))
                    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(img, class_names[cid], (x1, max(y1 - 5, 0)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

        ax.imshow(img)
        ax.set_title(img_path.name[:20], fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "yolo_format_sanity_check.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: yolo_format_sanity_check.png")


visualize_yolo_sample(
    YOLO_DIR / "images" / "train",
    YOLO_DIR / "labels" / "train",
    SELECTED_CLASSES,
    CLASS_COLORS,
    n=4
)

In [ ]:
#2.2.1
from ultralytics import YOLO
import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")

# Download base model (ImageNet pretrained backbone, no BDD100K weights)
model = YOLO(YOLO_MODEL)
print(f"Model loaded: {YOLO_MODEL}")
print(model.info())

In [ ]:
#2.2.2
import os
os.environ["WANDB_DISABLED"] = "true"  # disable wandb logging

RUN1_NAME = "yolo_no_aug"

model_no_aug = YOLO(YOLO_MODEL)

results_no_aug = model_no_aug.train(
    data      = str(YOLO_DIR / "data.yaml"),
    epochs    = EPOCHS,
    imgsz     = IMG_SIZE,
    batch     = BATCH_SIZE,
    optimizer = "SGD",
    lr0       = LR0,
    momentum  = MOMENTUM,
    weight_decay = 0.0005,
    patience  = PATIENCE,
    seed      = SEED,
    device = "0,1",
    project   = str(CKPT_DIR),
    name      = RUN1_NAME,
    exist_ok  = True,
    verbose   = True,

    # ── Augmentation OFF ──────────────────────────────────────────
    hsv_h     = 0.0,    # no hue jitter
    hsv_s     = 0.0,    # no saturation jitter
    hsv_v     = 0.0,    # no value jitter
    flipud    = 0.0,    # no vertical flip
    fliplr    = 0.0,    # no horizontal flip
    mosaic    = 0.0,    # no mosaic
    mixup     = 0.0,    # no mixup
    degrees   = 0.0,    # no rotation
    translate = 0.0,    # no translation
    scale     = 0.0,    # no scaling
    shear     = 0.0,    # no shear
    perspective = 0.0,  # no perspective
    close_mosaic = 0,   # keep off throughout
)

print("\nRun 1 (No Augmentation) training complete.")
print(f"Weights saved to: {CKPT_DIR}/{RUN1_NAME}/weights/")

In [ ]:
#2.2.3
import shutil

# Copy best weights to checkpoints root for easy access
best_weights_src = CKPT_DIR / RUN1_NAME / "weights" / "best.pt"
best_weights_dst = CKPT_DIR / "yolo_no_aug_best.pt"
shutil.copy(best_weights_src, best_weights_dst)
print(f"Best weights saved: {best_weights_dst}")

# Load and display results CSV
results_csv = CKPT_DIR / RUN1_NAME / "results.csv"
df_results_no_aug = pd.read_csv(results_csv)
df_results_no_aug.columns = df_results_no_aug.columns.str.strip()
print(f"\nTraining results shape: {df_results_no_aug.shape}")
print(df_results_no_aug.tail(5).to_string())

In [ ]:
#2.2.4
def plot_training_curves(df_results, run_name, save_path):
    df = df_results.copy()

    # Identify available columns
    print("Available columns:", df.columns.tolist())

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"Training Curves — {run_name}", fontsize=14, fontweight="bold")

    epochs = df["epoch"] if "epoch" in df.columns else range(len(df))

    # Loss curves
    for ax, col, title, color in [
        (axes[0][0], "train/box_loss",  "Train Box Loss",  "royalblue"),
        (axes[0][1], "train/cls_loss",  "Train Class Loss","tomato"),
        (axes[0][2], "train/dfl_loss",  "Train DFL Loss",  "green"),
        (axes[1][0], "val/box_loss",    "Val Box Loss",    "navy"),
        (axes[1][1], "val/cls_loss",    "Val Class Loss",  "darkred"),
        (axes[1][2], "val/dfl_loss",    "Val DFL Loss",    "darkgreen"),
    ]:
        if col in df.columns:
            axes_flat = ax
            axes_flat.plot(epochs, df[col], color=color, linewidth=2)
            axes_flat.set_title(title)
            axes_flat.set_xlabel("Epoch")
            axes_flat.set_ylabel("Loss")
            axes_flat.grid(True, alpha=0.3)
        else:
            ax.text(0.5, 0.5, f"'{col}'\nnot found",
                    ha="center", va="center", transform=ax.transAxes)
            ax.axis("off")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")


plot_training_curves(
    df_results_no_aug,
    run_name  = "YOLOv11 — No Augmentation",
    save_path = RESULTS_DIR / "yolo_no_aug_loss_curves.png"
)

# Also plot mAP curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("mAP Curves — No Augmentation", fontsize=13, fontweight="bold")

epochs = df_results_no_aug["epoch"] if "epoch" in df_results_no_aug.columns \
         else range(len(df_results_no_aug))

for ax, col, title, color in [
    (axes[0], "metrics/mAP50(B)",    "mAP@50",    "darkorange"),
    (axes[1], "metrics/mAP50-95(B)", "mAP@50-95", "purple"),
]:
    if col in df_results_no_aug.columns:
        ax.plot(epochs, df_results_no_aug[col], color=color, linewidth=2)
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(title)
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, f"'{col}'\nnot found",
                ha="center", va="center", transform=ax.transAxes)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "yolo_no_aug_map_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: yolo_no_aug_map_curves.png")

In [ ]:
#2.2.5
# Extract final epoch metrics
final = df_results_no_aug.iloc[-1]

# Safely extract metric values
def safe_get(df_row, col):
    return round(float(df_row[col]), 4) if col in df_row.index else None

metrics_no_aug = {
    "run"           : "no_augmentation",
    "epochs_trained": len(df_results_no_aug),
    "precision"     : safe_get(final, "metrics/precision(B)"),
    "recall"        : safe_get(final, "metrics/recall(B)"),
    "mAP50"         : safe_get(final, "metrics/mAP50(B)"),
    "mAP50_95"      : safe_get(final, "metrics/mAP50-95(B)"),
    "box_loss_final": safe_get(final, "val/box_loss"),
    "cls_loss_final": safe_get(final, "val/cls_loss"),
}

print("=== Run 1 Final Metrics (No Augmentation) ===")
for k, v in metrics_no_aug.items():
    print(f"  {k:<20}: {v}")

# Save for later comparison
df_metrics_no_aug = pd.DataFrame([metrics_no_aug])
df_metrics_no_aug.to_csv(RESULTS_DIR / "yolo_no_aug_metrics.csv", index=False)
print("\nSaved: yolo_no_aug_metrics.csv")

In [ ]:
#2.2.6
def run_inference_and_save(model_path, img_dir, save_dir, n=10, run_name="no_aug"):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    model    = YOLO(model_path)
    img_dir  = Path(img_dir)
    img_list = list(img_dir.glob("*.jpg"))[:n]

    if len(img_list) == 0:
        print("No images found in test dir.")
        return

    times = []
    fig, axes = plt.subplots(2, 5, figsize=(25, 10))
    fig.suptitle(f"Inference Results — {run_name}", fontsize=14, fontweight="bold")

    for i, (ax, img_path) in enumerate(zip(axes.flatten(), img_list)):
        import time
        t0      = time.time()
        results = model(str(img_path), verbose=False)
        t1      = time.time()
        times.append((t1 - t0) * 1000)  # ms

        # Render result
        result_img = results[0].plot()
        result_img = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)

        ax.imshow(result_img)
        ax.set_title(f"{img_path.name[:18]}\n{times[-1]:.1f}ms", fontsize=8)
        ax.axis("off")

        # Save individual image
        out_path = save_dir / f"{run_name}_inference_{i+1}.jpg"
        cv2.imwrite(str(out_path), cv2.cvtColor(result_img, cv2.COLOR_RGB2BGR))

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"yolo_{run_name}_inference.png",
                dpi=150, bbox_inches="tight")
    plt.show()

    avg_time = np.mean(times)
    print(f"Average inference time : {avg_time:.2f} ms/image")
    print(f"Approximate FPS        : {1000/avg_time:.1f}")
    print(f"Saved inference images to: {save_dir}")
    return avg_time


avg_time_no_aug = run_inference_and_save(
    model_path = CKPT_DIR / "yolo_no_aug_best.pt",
    img_dir    = YOLO_DIR / "images" / "test",
    save_dir   = RESULTS_DIR / "inference_no_aug",
    n          = 10,
    run_name   = "no_aug"
)

# Save inference speed to metrics
metrics_no_aug["inference_ms"] = round(avg_time_no_aug, 2)
pd.DataFrame([metrics_no_aug]).to_csv(
    RESULTS_DIR / "yolo_no_aug_metrics.csv", index=False)
print("Updated: yolo_no_aug_metrics.csv with inference speed")

**Yolo with Augmentation**

In [ ]:
#2.3.1
import os
os.environ["WANDB_DISABLED"] = "true"

RUN2_NAME = "yolo_with_aug"

model_with_aug = YOLO(YOLO_MODEL)

results_with_aug = model_with_aug.train(
    data         = str(YOLO_DIR / "data.yaml"),
    epochs       = EPOCHS,
    imgsz        = IMG_SIZE,
    batch        = BATCH_SIZE,
    optimizer    = "SGD",
    lr0          = LR0,
    momentum     = MOMENTUM,
    weight_decay = 0.0005,
    patience     = PATIENCE,
    seed         = SEED,
    device       = "0,1",        # T4 x2
    project      = str(CKPT_DIR),
    name         = RUN2_NAME,
    exist_ok     = True,
    verbose      = True,

    # ── Augmentation ON ───────────────────────────────────────────
    hsv_h        = 0.015,   # hue jitter
    hsv_s        = 0.7,     # saturation jitter
    hsv_v        = 0.4,     # value/brightness jitter
    fliplr       = 0.5,     # horizontal flip (mandatory, p=0.5)
    flipud       = 0.2,     # vertical flip (mandatory, p=0.2)
    mosaic       = 1.0,     # mosaic 4-image composite (mandatory)
    mixup        = 0.1,     # mixup (optional but encouraged)
    degrees      = 10.0,    # rotation ±10°
    translate    = 0.1,     # translation ±10%
    scale        = 0.5,     # scale ±50%
    shear        = 2.0,     # shear ±2°
    perspective  = 0.0001,  # perspective distortion
    close_mosaic = 10,      # disable mosaic last 10 epochs for stability
)

print("\nRun 2 (With Augmentation) training complete.")
print(f"Weights saved to: {CKPT_DIR}/{RUN2_NAME}/weights/")

In [ ]:
#2.3.2
import shutil

# Copy best weights
best_weights_src = CKPT_DIR / RUN2_NAME / "weights" / "best.pt"
best_weights_dst = CKPT_DIR / "yolo_with_aug_best.pt"
shutil.copy(best_weights_src, best_weights_dst)
print(f"Best weights saved: {best_weights_dst}")

# Load results CSV
results_csv = CKPT_DIR / RUN2_NAME / "results.csv"
df_results_with_aug = pd.read_csv(results_csv)
df_results_with_aug.columns = df_results_with_aug.columns.str.strip()
print(f"\nTraining results shape: {df_results_with_aug.shape}")
print(df_results_with_aug.tail(5).to_string())

In [ ]:
#2.3.3
plot_training_curves(
    df_results_with_aug,
    run_name  = "YOLOv11 — With Augmentation",
    save_path = RESULTS_DIR / "yolo_with_aug_loss_curves.png"
)

# mAP curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("mAP Curves — With Augmentation", fontsize=13, fontweight="bold")

epochs = df_results_with_aug["epoch"] if "epoch" in df_results_with_aug.columns \
         else range(len(df_results_with_aug))

for ax, col, title, color in [
    (axes[0], "metrics/mAP50(B)",    "mAP@50",    "darkorange"),
    (axes[1], "metrics/mAP50-95(B)", "mAP@50-95", "purple"),
]:
    if col in df_results_with_aug.columns:
        ax.plot(epochs, df_results_with_aug[col], color=color, linewidth=2)
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(title)
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, f"'{col}'\nnot found",
                ha="center", va="center", transform=ax.transAxes)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "yolo_with_aug_map_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: yolo_with_aug_map_curves.png")

In [ ]:
#2.3.4
final2 = df_results_with_aug.iloc[-1]

metrics_with_aug = {
    "run"           : "with_augmentation",
    "epochs_trained": len(df_results_with_aug),
    "precision"     : safe_get(final2, "metrics/precision(B)"),
    "recall"        : safe_get(final2, "metrics/recall(B)"),
    "mAP50"         : safe_get(final2, "metrics/mAP50(B)"),
    "mAP50_95"      : safe_get(final2, "metrics/mAP50-95(B)"),
    "box_loss_final": safe_get(final2, "val/box_loss"),
    "cls_loss_final": safe_get(final2, "val/cls_loss"),
}

print("=== Run 2 Final Metrics (With Augmentation) ===")
for k, v in metrics_with_aug.items():
    print(f"  {k:<20}: {v}")

df_metrics_with_aug = pd.DataFrame([metrics_with_aug])
df_metrics_with_aug.to_csv(RESULTS_DIR / "yolo_with_aug_metrics.csv", index=False)
print("\nSaved: yolo_with_aug_metrics.csv")

In [ ]:
#2.3.5
avg_time_with_aug = run_inference_and_save(
    model_path = CKPT_DIR / "yolo_with_aug_best.pt",
    img_dir    = YOLO_DIR / "images" / "test",
    save_dir   = RESULTS_DIR / "inference_with_aug",
    n          = 10,
    run_name   = "with_aug"
)

metrics_with_aug["inference_ms"] = round(avg_time_with_aug, 2)
pd.DataFrame([metrics_with_aug]).to_csv(
    RESULTS_DIR / "yolo_with_aug_metrics.csv", index=False)
print("Updated: yolo_with_aug_metrics.csv with inference speed")

In [ ]:
#2.3.6
# Reload no_aug metrics in case session restarted
df_no_aug_saved   = pd.read_csv(RESULTS_DIR / "yolo_no_aug_metrics.csv")
df_with_aug_saved = pd.read_csv(RESULTS_DIR / "yolo_with_aug_metrics.csv")

df_comparison = pd.concat([df_no_aug_saved, df_with_aug_saved], ignore_index=True)

# Compute deltas
metrics_cols = ["precision", "recall", "mAP50", "mAP50_95"]
delta_row    = {"run": "delta (aug - no_aug)"}
for col in metrics_cols:
    v_aug    = df_with_aug_saved[col].values[0]
    v_no_aug = df_no_aug_saved[col].values[0]
    delta    = round(v_aug - v_no_aug, 4)
    delta_row[col] = f"+{delta}" if delta >= 0 else str(delta)

df_comparison = pd.concat(
    [df_comparison, pd.DataFrame([delta_row])],
    ignore_index=True
)

print("=== Augmentation Comparison Table ===\n")
print(df_comparison[["run", "precision", "recall",
                      "mAP50", "mAP50_95",
                      "epochs_trained", "inference_ms"]].to_string(index=False))

df_comparison.to_csv(RESULTS_DIR / "yolo_augmentation_comparison.csv", index=False)
print("\nSaved: yolo_augmentation_comparison.csv")

# Visual comparison bar chart
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle("YOLOv11 — Augmentation vs No Augmentation", fontsize=13, fontweight="bold")

runs   = ["No Augmentation", "With Augmentation"]
colors = ["steelblue", "tomato"]

for ax, metric in zip(axes, metrics_cols):
    values = [
        float(df_no_aug_saved[metric].values[0]),
        float(df_with_aug_saved[metric].values[0]),
    ]
    bars = ax.bar(runs, values, color=colors, edgecolor="black", width=0.5)
    ax.set_title(metric.upper())
    ax.set_ylim(0, max(values) * 1.25)
    ax.set_ylabel("Score")
    for bar, v in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                v + 0.005, f"{v:.4f}",
                ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "yolo_augmentation_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: yolo_augmentation_comparison.png")

**YOLO Evaluation**

In [ ]:
#2.4.1
from ultralytics import YOLO

# Load best augmented model for final evaluation
model_eval = YOLO(str(CKPT_DIR / "yolo_with_aug_best.pt"))

# Run validation on test split
test_results = model_eval.val(
    data     = str(YOLO_DIR / "data.yaml"),
    split    = "test",
    imgsz    = IMG_SIZE,
    batch    = BATCH_SIZE,
    device   = 0,
    verbose  = True,
)

# Extract per-class metrics
per_class_metrics = []
for i, cls_name in enumerate(SELECTED_CLASSES):
    per_class_metrics.append({
        "class"    : cls_name,
        "precision": round(float(test_results.box.p[i]), 4),
        "recall"   : round(float(test_results.box.r[i]), 4),
        "mAP50"    : round(float(test_results.box.ap50[i]), 4),
        "mAP50_95" : round(float(test_results.box.ap[i]), 4),
    })

# Add mean row
per_class_metrics.append({
    "class"    : "MEAN",
    "precision": round(float(test_results.box.mp), 4),
    "recall"   : round(float(test_results.box.mr), 4),
    "mAP50"    : round(float(test_results.box.map50), 4),
    "mAP50_95" : round(float(test_results.box.map), 4),
})

df_per_class = pd.DataFrame(per_class_metrics)
print("=== Per-Class Metrics (With Augmentation Model — Test Split) ===\n")
print(df_per_class.to_string(index=False))

df_per_class.to_csv(RESULTS_DIR / "detection_results.csv", index=False)
print("\nSaved: detection_results.csv")

# Bar chart
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Per-Class Detection Metrics — YOLOv11 (With Augmentation)",
             fontsize=14, fontweight="bold")

df_plot = df_per_class[df_per_class["class"] != "MEAN"]
colors  = plt.cm.tab10(np.linspace(0, 1, len(df_plot)))

for ax, metric in zip(axes.flatten(),
                      ["precision", "recall", "mAP50", "mAP50_95"]):
    bars = ax.bar(df_plot["class"], df_plot[metric],
                  color=colors, edgecolor="black")
    ax.set_title(metric.upper())
    ax.set_xlabel("Class")
    ax.set_ylabel("Score")
    ax.tick_params(axis="x", rotation=45)
    ax.set_ylim(0, 1.0)
    mean_val = df_per_class[df_per_class["class"] == "MEAN"][metric].values[0]
    ax.axhline(mean_val, color="red", linestyle="--",
               linewidth=1.5, label=f"Mean={mean_val:.3f}")
    ax.legend(fontsize=8)
    for bar, v in zip(bars, df_plot[metric]):
        ax.text(bar.get_x() + bar.get_width() / 2,
                v + 0.01, f"{v:.3f}",
                ha="center", va="bottom", fontsize=7, rotation=45)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "yolo_per_class_metrics.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: yolo_per_class_metrics.png")

In [ ]:
#2.4.2
from ultralytics import YOLO
import matplotlib.pyplot as plt
import numpy as np

# Use no_aug model too for comparison
model_no_aug_eval  = YOLO(str(CKPT_DIR / "yolo_no_aug_best.pt"))
model_with_aug_eval = YOLO(str(CKPT_DIR / "yolo_with_aug_best.pt"))

# Run val to get PR curve data for both models
val_no_aug   = model_no_aug_eval.val(
    data=str(YOLO_DIR / "data.yaml"), split="test",
    imgsz=IMG_SIZE, batch=BATCH_SIZE, device=0, verbose=False)

val_with_aug = model_with_aug_eval.val(
    data=str(YOLO_DIR / "data.yaml"), split="test",
    imgsz=IMG_SIZE, batch=BATCH_SIZE, device=0, verbose=False)

# Plot PR curves per class
n_classes = NUM_CLASSES
fig, axes = plt.subplots(2, 5, figsize=(25, 10))
fig.suptitle("Precision-Recall Curves Per Class\n(Blue=No Aug, Red=With Aug)",
             fontsize=14, fontweight="bold")

for i, (ax, cls_name) in enumerate(zip(axes.flatten(), SELECTED_CLASSES)):
    try:
        # No aug
        p_no  = val_no_aug.box.p[i]
        r_no  = val_no_aug.box.r[i]
        # With aug
        p_aug = val_with_aug.box.p[i]
        r_aug = val_with_aug.box.r[i]

        # Get full PR curve data if available
        if hasattr(val_no_aug.box, 'curves_results'):
            pr_data_no  = val_no_aug.box.curves_results
            pr_data_aug = val_with_aug.box.curves_results
            ax.plot(pr_data_no[1][i],  pr_data_no[0][i],
                    color="royalblue", linewidth=2, label=f"No Aug (AP={val_no_aug.box.ap50[i]:.3f})")
            ax.plot(pr_data_aug[1][i], pr_data_aug[0][i],
                    color="tomato",    linewidth=2, label=f"With Aug (AP={val_with_aug.box.ap50[i]:.3f})")
        else:
            # Fallback: plot single points
            ax.scatter([r_no],  [p_no],  color="royalblue", s=100, zorder=5,
                       label=f"No Aug  P={p_no:.3f} R={r_no:.3f}")
            ax.scatter([r_aug], [p_aug], color="tomato",    s=100, zorder=5,
                       label=f"Aug     P={p_aug:.3f} R={r_aug:.3f}")
            ax.plot([0, r_no,  1], [1, p_no,  0], color="royalblue",
                    linewidth=1.5, linestyle="--", alpha=0.7)
            ax.plot([0, r_aug, 1], [1, p_aug, 0], color="tomato",
                    linewidth=1.5, linestyle="--", alpha=0.7)

    except Exception as e:
        ax.text(0.5, 0.5, f"No data\n{str(e)[:30]}",
                ha="center", va="center", transform=ax.transAxes, fontsize=8)

    ax.set_title(cls_name, fontsize=10, fontweight="bold")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "yolo_pr_curves_per_class.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: yolo_pr_curves_per_class.png")

In [ ]:
#2.4.3
from ultralytics import YOLO

# Generate confusion matrix using ultralytics built-in
model_cm = YOLO(str(CKPT_DIR / "yolo_with_aug_best.pt"))

# Run val with plots=True to auto-generate confusion matrix
val_cm = model_cm.val(
    data     = str(YOLO_DIR / "data.yaml"),
    split    = "test",
    imgsz    = IMG_SIZE,
    batch    = BATCH_SIZE,
    device   = 0,
    plots    = True,             # generates confusion matrix automatically
    save_dir = str(RESULTS_DIR / "confusion_matrix"),
    verbose  = False,
)

# Copy confusion matrix plot to results
import shutil, glob
cm_dir = RESULTS_DIR / "confusion_matrix"
cm_files = list(cm_dir.glob("confusion_matrix*.png")) if cm_dir.exists() else []

# Also search in val run dir
val_run_dir = Path(str(CKPT_DIR / "yolo_with_aug"))
cm_files   += list(val_run_dir.rglob("confusion_matrix*.png"))

if cm_files:
    shutil.copy(cm_files[0], RESULTS_DIR / "yolo_confusion_matrix.png")
    print(f"Confusion matrix copied to: {RESULTS_DIR}/yolo_confusion_matrix.png")

    img = Image.open(cm_files[0])
    plt.figure(figsize=(12, 10))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Confusion Matrix — YOLOv11 With Augmentation (Test Set)",
              fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "yolo_confusion_matrix.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Confusion matrix PNG not found in expected dirs.")
    print("Searching everywhere...")
    found = list(Path(CKPT_DIR).rglob("confusion_matrix*.png"))
    print(f"Found: {found}")
    if found:
        shutil.copy(found[0], RESULTS_DIR / "yolo_confusion_matrix.png")
        img = Image.open(found[0])
        plt.figure(figsize=(12, 10))
        plt.imshow(img)
        plt.axis("off")
        plt.tight_layout()
        plt.show()

print("\nPhase 2 Evaluation Complete.")
print("All detection deliverables saved to:", RESULTS_DIR)